# MiVOLO עם Florence-2 לזיהוי גיל ומגדר

מחברת זו מדגימה כיצד להשתמש במודל MiVOLO להערכת גיל ומגדר, כאשר שלב זיהוי האובייקטים (אנשים ופנים) מתבצע באמצעות המודל Florence-2 של מיקרוסופט.

## 1. התקנת ספריות נדרשות

תחילה, נתקין את הספריות הדרושות. יש להריץ תא זה רק פעם אחת.

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
!pip install transformers accelerate einops timm opencv-python matplotlib Pillow scipy ultralytics
# אם הקוד של mivolo לא מותקן כחבילה, נדאג שפייתון ימצא אותו (שנה נתיב אם צריך)
# import sys
# sys.path.append('.') # הוספת התיקייה הנוכחית לנתיבי החיפוש

## 2. יבוא ספריות ומודולים

נייבא את כל הספריות והמודולים הנדרשים מהקוד שלנו וספריות חיצוניות.

In [1]:
import os
import cv2  # ייבוא OpenCV
import torch
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
import types # ליצירת אובייקט קונפיגורציה פשוט

# ייבוא מהקוד של mivolo (ודא שהנתיבים נכונים או שהחבילה מותקנת)
try:
    from mivolo.predictor import Predictor
    from mivolo.model.florence2_detector import Florence2Detector # ודא שהשינויים שביצענו קיימים בקובץ זה
    from mivolo.model.mi_volo import MiVOLO
    from mivolo.structures import PersonAndFaceResult # ודא שהשינויים שביצענו קיימים בקובץ זה
    print("MiVOLO modules imported successfully.")
except ImportError as e:
    print(f"Error importing MiVOLO modules: {e}")
    print("Please ensure the mivolo directory is in your Python path or installed.")
    # לדוגמה, אם מריצים מהתיקייה הראשית של הפרויקט:
    # import sys
    # sys.path.insert(0, os.path.abspath('.')) 
    # from mivolo.predictor import Predictor
    # ... וכן הלאה ...
except Exception as e:
    print(f"An unexpected error occurred during import: {e}")

Error importing MiVOLO modules: cannot import name 'remap_checkpoint' from 'timm.models._helpers' (c:\Users\me\AppData\Local\Programs\Python\Python310\lib\site-packages\timm\models\_helpers.py)
Please ensure the mivolo directory is in your Python path or installed.


## 3. הגדרות וקונפיגורציה

נגדיר את הנתיבים לקבצים, מזהה המודל של Florence-2, והגדרות נוספות.

In [3]:
# --- הגדרות עיקריות --- 
INPUT_IMAGE_PATH = "images/1.jpg"  # שנה לנתיב התמונה שלך
OUTPUT_DIR = "output_notebook"       # תיקייה לשמירת התוצאות
MI_volo_CHECKPOINT_PATH = "models/model_imdb_cross_person_4.22_99.46.pth.tar" # נתיב לקובץ המשקולות של MiVOLO
FLORENCE2_MODEL_ID = 'microsoft/Florence-2-base' # מזהה המודל ב-Hugging Face

# --- הגדרות נוספות --- 
DEVICE = "cpu"  # שנה ל-"cuda" אם יש לך GPU זמין ותומך
WITH_PERSONS = True # האם להשתמש גם בקרופ של הגוף (דורש מודל MiVOLO מתאים)
DISABLE_FACES = False # האם להתעלם מהפנים ולהשתמש רק בגוף (דורש WITH_PERSONS=True)
DRAW_OUTPUT = True  # האם לצייר את התוצאות על התמונה
VERBOSE = True    # האם להדפיס לוגים מפורטים
HALF_PRECISION = False # האם לנסות להשתמש ב-half precision (לא מומלץ ל-CPU, דורש בדיקה ב-GPU)

# --- יצירת תיקיית פלט --- 
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- יצירת אובייקט קונפיגורציה דמוי argparse --- 
config = types.SimpleNamespace()
config.input = INPUT_IMAGE_PATH
config.output = OUTPUT_DIR
config.checkpoint = MI_volo_CHECKPOINT_PATH
config.florence_model_id = FLORENCE2_MODEL_ID # הוספנו פרמטר זה במקום detector_weights
config.device = DEVICE
config.with_persons = WITH_PERSONS
config.disable_faces = DISABLE_FACES
config.draw = DRAW_OUTPUT
config.half = HALF_PRECISION

print("Configuration set:")
print(f"Input Image: {config.input}")
print(f"Output Directory: {config.output}")
print(f"MiVOLO Checkpoint: {config.checkpoint}")
print(f"Florence-2 Model ID: {config.florence_model_id}")
print(f"Device: {config.device}")
print(f"Use Person Crops: {config.with_persons}")
print(f"Disable Face Crops: {config.disable_faces}")
print(f"Draw Output: {config.draw}")
print(f"Use Half Precision: {config.half}")

Configuration set:
Input Image: images/1.jpg
Output Directory: output_notebook
MiVOLO Checkpoint: models/model_imdb_cross_person_4.22_99.46.pth.tar
Florence-2 Model ID: microsoft/Florence-2-base
Device: cpu
Use Person Crops: True
Disable Face Crops: False
Draw Output: True
Use Half Precision: False


## 4. אתחול ה-Predictor

ניצור אובייקט `Predictor` שיאתחל את מודל Florence-2 ואת מודל MiVOLO. שלב זה עשוי לקחת זמן, במיוחד בהרצה ראשונה כאשר המודלים יורדים.

In [4]:
try:
    # העברנו את אובייקט הקונפיגורציה שיצרנו
    predictor = Predictor(config, verbose=VERBOSE)
    print("Predictor initialized successfully.")
except NameError as ne:
    print(f"A required class or function might be missing: {ne}")
    print("Ensure all necessary imports completed successfully.")
except FileNotFoundError as fnfe:
    print(f"Error: Model file not found: {fnfe}")
    print("Please check the checkpoint path and Florence-2 model ID.")
except Exception as e:
    print(f"An unexpected error occurred during Predictor initialization: {e}")
    # ניתן להוסיף כאן טיפול בשגיאות נוספות לפי הצורך

A required class or function might be missing: name 'Predictor' is not defined
Ensure all necessary imports completed successfully.


## 5. טעינת התמונה והרצת הזיהוי

נטען את תמונת הקלט ונריץ את תהליך הזיהוי והערכת הגיל/מגדר.

In [6]:
# טעינת התמונה באמצעות OpenCV (מחזיר numpy array בפורמט BGR)
print(f"Loading image: {config.input}")
image_cv2 = cv2.imread(config.input)

if image_cv2 is None:
    print(f"Error: Could not load image from {config.input}")
else:
    print("Image loaded successfully. Running recognition...")
    # הרצת הזיהוי וההערכה
    # הפונקציה מצפה לקבל numpy array (כמו ש-cv2 מחזיר)
    try:
        detected_objects, output_image_annotated = predictor.recognize(image_cv2)
        
        if detected_objects:
            print(f"Recognition complete. Found {detected_objects.n_objects} relevant objects (persons/faces)." )
            # ניתן להדפיס פרטים נוספים על האובייקטים שזוהו אם רוצים
            # for i in range(detected_objects.n_objects):
            #     label = detected_objects.labels[i]
            #     age = detected_objects.ages[i]
            #     gender = detected_objects.genders[i]
            #     print(f"  Object {i}: Label={label}, Age={age}, Gender={gender}")
        else:
             print("Recognition complete, but no relevant objects were passed to MiVOLO or detected.")
             
        # שמירת התמונה עם התוצאות (אם הציור הופעל)
        if config.draw and output_image_annotated is not None:
            output_filename = os.path.join(config.output, f"out_notebook_{os.path.basename(config.input)}")
            try:
                # נמיר ל-RGB לפני השמירה אם נרצה פורמט סטנדרטי יותר, או נשמור כ-BGR
                # output_image_rgb = cv2.cvtColor(output_image_annotated, cv2.COLOR_BGR2RGB)
                cv2.imwrite(output_filename, output_image_annotated) 
                print(f"Annotated image saved to: {output_filename}")
            except Exception as e:
                print(f"Error saving annotated image: {e}")
        elif config.draw:
             print("Drawing was enabled, but no annotated image was returned.")
             
    except AttributeError as ae:
         print(f"Attribute Error during recognition: {ae}. This might indicate an issue with class structures (e.g., PersonAndFaceResult) or method calls.")
    except Exception as e:
         print(f"An error occurred during recognition: {e}")
         import traceback
         traceback.print_exc() # הדפסת traceback מלא לדיבוג

Loading image: images/1.jpg
Image loaded successfully. Running recognition...
An error occurred during recognition: name 'predictor' is not defined


Traceback (most recent call last):
  File "C:\Users\me\AppData\Local\Temp\ipykernel_8412\2507020713.py", line 12, in <module>
    detected_objects, output_image_annotated = predictor.recognize(image_cv2)
NameError: name 'predictor' is not defined


## 6. הצגת התוצאות

נציג את התמונה המקורית ואת התמונה עם התוצאות שצוירו עליה (אם הציור הופעל).

In [7]:
if image_cv2 is not None:
    # המרה מ-BGR (OpenCV) ל-RGB (Matplotlib)
    image_rgb = cv2.cvtColor(image_cv2, cv2.COLOR_BGR2RGB)
    
    if config.draw and output_image_annotated is not None:
        output_image_rgb = cv2.cvtColor(output_image_annotated, cv2.COLOR_BGR2RGB)
        
        # הצגת התמונות זו לצד זו
        fig, axes = plt.subplots(1, 2, figsize=(15, 7))
        
        axes[0].imshow(image_rgb)
        axes[0].set_title('Original Image')
        axes[0].axis('off')
        
        axes[1].imshow(output_image_rgb)
        axes[1].set_title('Annotated Image (Age/Gender)')
        axes[1].axis('off')
        
        plt.tight_layout()
        plt.show()
    else:
        # הצגת התמונה המקורית בלבד
        plt.figure(figsize=(8, 8))
        plt.imshow(image_rgb)
        plt.title('Original Image (No annotation drawing)')
        plt.axis('off')
        plt.show()
else:
    print("Cannot display images because the original image failed to load.")

NameError: name 'output_image_annotated' is not defined